# 🧠 Global Mental Health & Depression Rates
## Data Story — Team Project | Tec de Monterrey
**Dataset:** Our World in Data — Mental Health Disorders (1990–2017)  
**Big Idea:** *Depression is the world's most underestimated health crisis — and the countries that ignore it today will pay the highest cost tomorrow.*


In [ ]:
# ── Imports & global style settings ────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
import warnings
warnings.filterwarnings("ignore")

# ── Color palette: white background + 2 accent colors ──────────
# ACCENT (cyan) = relevant / highlighted data
# GRAY          = everything else (muted)
BG     = "#ffffff"        # white background
ACCENT = "#0096c7"        # deep cyan — relevant data (darker for white BG)
GRAY   = "#d0d5dd"        # light gray — muted data
RED    = "#e63946"        # warning / negative accent
TICK   = "#6c757d"        # axis tick color
TEXT   = "#1a2540"        # main text color (dark navy on white)
GRID   = "#f0f2f5"        # subtle grid lines

plt.rcParams.update({
    "figure.facecolor": BG,
    "axes.facecolor":   BG,
    "axes.edgecolor":   "#e9ecef",
    "axes.labelcolor":  TICK,
    "xtick.color":      TICK,
    "ytick.color":      TICK,
    "text.color":       TEXT,
    "grid.color":       GRID,
    "grid.linewidth":   0.8,
    "font.family":      "sans-serif",
    "font.size":        10,
})
print("✅ Libraries loaded — white background, cyan accent.")


## 1. Load & Clean Dataset

In [ ]:
# ── Load dataset ────────────────────────────────────────────────
df_raw = pd.read_csv("Mental_health_Depression_disorder_Data.csv", low_memory=False)

df = df_raw.copy()
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df["Depression (%)"] = pd.to_numeric(df["Depression (%)"], errors="coerce")
df["Anxiety disorders (%)"] = pd.to_numeric(df["Anxiety disorders (%)"], errors="coerce")
df = df.dropna(subset=["Year", "Depression (%)"])
df["Year"] = df["Year"].astype(int)

# Countries have 3-letter Code; regions do not
countries_df = df[df["Code"].notna() & (df["Code"] != "") & (df["Code"].str.len() == 3)].copy()
regions_df   = df[~(df["Code"].notna() & (df["Code"] != "") & (df["Code"].str.len() == 3))].copy()

print(f"Dataset: {df.shape[0]:,} rows | {df['Entity'].nunique()} entities")
print(f"Countries: {countries_df['Entity'].nunique()} | Years: {df['Year'].min()}–{df['Year'].max()}")
df.head(3)


## 2. Chart 1 — Global Average Depression Rate Over Time
**Visual type:** Line chart — best for trends over time.  
**Design decision:** Single cyan line on white background. Peak year annotated.  
**Preattentive attribute:** Color (cyan against muted gray grid).


In [ ]:
# ── CHART 1: Global trend line ──────────────────────────────────
global_trend = countries_df.groupby("Year")["Depression (%)"].mean().reset_index()
peak = global_trend.loc[global_trend["Depression (%)"].idxmax()]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(global_trend["Year"], global_trend["Depression (%)"],
        color=ACCENT, linewidth=2.5, zorder=3)
ax.fill_between(global_trend["Year"], global_trend["Depression (%)"],
                alpha=0.10, color=ACCENT)

ax.annotate(
    f"Peak: {peak['Depression (%)']:.2f}%\n(Year {int(peak['Year'])})",
    xy=(peak["Year"], peak["Depression (%)"]),
    xytext=(peak["Year"] - 6, peak["Depression (%)"] + 0.05),
    fontsize=9, color=ACCENT, fontweight="600",
    arrowprops=dict(arrowstyle="->", color=ACCENT, lw=1.2)
)

ax.set_title("Global Average Depression Rate Has Been Rising (1990–2017)",
             fontsize=13, color=TEXT, pad=14, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Avg. Share of Population (%)")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f%%"))
ax.grid(axis="y")
ax.set_xlim(1990, 2017)
for spine in ["top","right"]: ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.savefig("chart1_global_trend.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print("Saved: chart1_global_trend.png")


## 3. Chart 2 — Top 10 Countries by Depression Rate
**Visual type:** Horizontal bar chart — best for category comparison.  
**Design decision:** Only #1 country in cyan, the rest muted gray.  
**Preattentive attribute:** Color contrast (1 vs many).


In [ ]:
# ── CHART 2: Top 10 countries — latest year ─────────────────────
latest_year = countries_df["Year"].max()
latest = countries_df[countries_df["Year"] == latest_year].copy()
top10 = latest.nlargest(10, "Depression (%)").sort_values("Depression (%)", ascending=True)

fig, ax = plt.subplots(figsize=(11, 5.5))
bar_colors = [ACCENT if i == len(top10)-1 else GRAY for i in range(len(top10))]

bars = ax.barh(top10["Entity"], top10["Depression (%)"],
               color=bar_colors, height=0.6, zorder=2)

for bar, val in zip(bars, top10["Depression (%)"]):
    ax.text(val + 0.02, bar.get_y() + bar.get_height()/2,
            f"{val:.2f}%", va="center", fontsize=9, color=TICK)

ax.set_title(f"Countries with Highest Depression Rates ({latest_year})",
             fontsize=13, color=TEXT, pad=14, fontweight="bold")
ax.set_xlabel("Share of Population (%)")
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
ax.set_xlim(0, top10["Depression (%)"].max() * 1.2)
ax.grid(axis="x")
for spine in ["top","right"]: ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.savefig("chart2_top_countries.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print("Saved: chart2_top_countries.png")


## 4. Chart 3 — Depression Rate Heatmap
**Visual type:** Heatmap — patterns across two dimensions.  
**Design decision:** White → cyan colormap. Brighter = higher rate.  
**Preattentive attribute:** Color intensity.


In [ ]:
# ── CHART 3: Heatmap ───────────────────────────────────────────
reg_data = regions_df[regions_df["Entity"].str.contains(
    "Asia|Europe|Africa|America|Pacific|Caribbean|Middle East",
    case=False, na=False
)].copy()

if len(reg_data) < 5:
    reg_data = countries_df.copy()
    reg_data["Region"] = pd.cut(reg_data["Depression (%)"], bins=5,
                                 labels=["Very Low","Low","Medium","High","Very High"])
    hm_data = reg_data.copy()
    hm_data["Decade"] = (hm_data["Year"] // 5) * 5
    hm = hm_data.groupby(["Region","Decade"])["Depression (%)"].mean().unstack()
else:
    reg_data["Decade"] = (reg_data["Year"] // 5) * 5
    hm = reg_data.groupby(["Entity","Decade"])["Depression (%)"].mean().unstack()
    hm = hm.head(10)

cmap = LinearSegmentedColormap.from_list("mh", ["#ffffff", "#caf0f8", "#48cae4", ACCENT])

fig, ax = plt.subplots(figsize=(12, 4.5))
sns.heatmap(hm, ax=ax, cmap=cmap, linewidths=0.8, linecolor="#ffffff",
            annot=True, fmt=".2f",
            annot_kws={"size":9, "color":TEXT},
            cbar_kws={"shrink":0.6})
ax.set_title("Average Depression Rate (%) — Regions Over Time\nBrighter = Higher Burden",
             fontsize=13, color=TEXT, pad=14, fontweight="bold")
ax.set_xlabel("Period"); ax.set_ylabel("")
ax.tick_params(colors=TICK, labelsize=9)
for spine in ax.spines.values(): spine.set_visible(False)
plt.tight_layout()
plt.savefig("chart3_heatmap.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print("Saved: chart3_heatmap.png")


## 5. Chart 4 — Country Trajectories: One vs. Many
**Visual type:** Multi-line chart — compares trends across entities.  
**Design decision:** Highest-burden country in cyan, others muted gray.  
**Preattentive attribute:** Color + line weight.


In [ ]:
# ── CHART 4: Multi-line country comparison ──────────────────────
top_countries = (countries_df.groupby("Entity")["Depression (%)"]
                 .mean().nlargest(12).index.tolist())
highlight = top_countries[0]

fig, ax = plt.subplots(figsize=(12, 5.5))

for country in top_countries:
    cdata = countries_df[countries_df["Entity"] == country].sort_values("Year")
    if country == highlight:
        ax.plot(cdata["Year"], cdata["Depression (%)"],
                color=ACCENT, linewidth=2.8, zorder=3, label=country)
        ax.text(cdata["Year"].max() + 0.3,
                cdata["Depression (%)"].iloc[-1],
                country, fontsize=8.5, color=ACCENT,
                va="center", fontweight="600")
    else:
        ax.plot(cdata["Year"], cdata["Depression (%)"],
                color=GRAY, linewidth=1.1, alpha=0.85, zorder=2)

ax.set_title(f"Depression Rate Trends — {highlight} Stands Out",
             fontsize=13, color=TEXT, pad=14, fontweight="bold")
ax.set_xlabel("Year"); ax.set_ylabel("Share of Population (%)")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
ax.set_xlim(1990, 2017)
ax.grid(axis="y")
for spine in ["top","right"]: ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.savefig("chart4_country_lines.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print("Saved: chart4_country_lines.png")


## 6. Chart 5 — Slope Graph: 1990 vs 2017
**Visual type:** Slope graph — direction & magnitude of change.  
**Design decision:** Worsening countries in cyan, improving in gray.  
**Preattentive attribute:** Position + color.


In [ ]:
# ── CHART 5: Slope graph ───────────────────────────────────────
start_y, end_y = 1990, 2017

slope = countries_df[countries_df["Year"].isin([start_y, end_y])].copy()
slope_piv = slope.pivot_table(index="Entity", columns="Year",
                               values="Depression (%)").dropna()
slope_piv["change"] = slope_piv[end_y] - slope_piv[start_y]
slope_piv = slope_piv.reindex(
    slope_piv["change"].abs().nlargest(14).index
).sort_values("change", ascending=False)

fig, ax = plt.subplots(figsize=(8, 7))

for country, row in slope_piv.iterrows():
    increased = row["change"] > 0
    color = ACCENT if increased else GRAY
    lw    = 2.2 if abs(row["change"]) > slope_piv["change"].abs().quantile(0.7) else 1.2
    ax.plot([0, 1], [row[start_y], row[end_y]], color=color, linewidth=lw, alpha=0.9)
    ax.text(-0.04, row[start_y], country, ha="right", fontsize=7.5,
            color=color, fontweight="600" if increased else "400")
    ax.text(1.04, row[end_y], f"{row[end_y]:.1f}%", ha="left", fontsize=7.5,
            color=color, fontweight="600" if increased else "400")

ax.set_xlim(-0.5, 1.6)
ax.set_xticks([0, 1])
ax.set_xticklabels(["1990", "2017"], fontsize=11, color=TEXT, fontweight="600")
ax.set_ylabel("Share of Population (%)")
ax.set_title("Depression Rates: How Countries Changed from 1990 to 2017\nCyan = Worsened · Gray = Improved/Stable",
             fontsize=12, color=TEXT, pad=14, fontweight="bold")
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.1f%%"))
ax.grid(axis="y", alpha=0.5)
for spine in ["top","right","bottom"]: ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.savefig("chart5_slope_graph.png", dpi=150, bbox_inches="tight", facecolor=BG)
plt.show()
print("Saved: chart5_slope_graph.png")


## 7. Chart 6 — Interactive: Depression vs Anxiety (Plotly)
**Visual type:** Scatter plot — relationship between two disorders.  
**Design decision:** Above-average depression in cyan, rest gray.  
**Preattentive attribute:** Color + size.


In [ ]:
# ── CHART 6: Interactive Plotly scatter ─────────────────────────
latest2 = countries_df[countries_df["Year"] == countries_df["Year"].max()].dropna(
    subset=["Depression (%)","Anxiety disorders (%)"]
).copy()

avg_dep = latest2["Depression (%)"].mean()
latest2["highlight"] = latest2["Depression (%)"] > avg_dep

fig6 = go.Figure()

bg = latest2[~latest2["highlight"]]
fig6.add_trace(go.Scatter(
    x=bg["Anxiety disorders (%)"], y=bg["Depression (%)"],
    mode="markers", name="Below avg",
    marker=dict(color=GRAY, size=8, opacity=0.7),
    text=bg["Entity"],
    hovertemplate="<b>%{text}</b><br>Anxiety: %{x:.2f}%<br>Depression: %{y:.2f}%<extra></extra>"
))

hi = latest2[latest2["highlight"]]
fig6.add_trace(go.Scatter(
    x=hi["Anxiety disorders (%)"], y=hi["Depression (%)"],
    mode="markers", name="Above avg depression",
    marker=dict(color=ACCENT, size=11, opacity=0.95,
                line=dict(color="#005f73", width=1)),
    text=hi["Entity"],
    hovertemplate="<b>%{text}</b><br>Anxiety: %{x:.2f}%<br>Depression: %{y:.2f}%<extra></extra>"
))

fig6.add_hline(y=avg_dep, line_dash="dot", line_color="#adb5bd",
               annotation_text=f"Avg depression {avg_dep:.2f}%",
               annotation_font_color="#6c757d")

fig6.update_layout(
    title=dict(text="Depression vs. Anxiety Rates by Country — Cyan = Above Average Depression",
               font=dict(size=13, color="#1a2540")),
    plot_bgcolor="#ffffff", paper_bgcolor="#ffffff",
    font=dict(family="sans-serif", color="#6c757d"),
    xaxis=dict(title="Anxiety Disorders (%)", showgrid=True, gridcolor="#f0f2f5", color="#6c757d"),
    yaxis=dict(title="Depression (%)", showgrid=True, gridcolor="#f0f2f5", color="#6c757d"),
    legend=dict(bgcolor="rgba(0,0,0,0)", font=dict(color="#6c757d")),
    height=480
)
fig6.write_html("chart6_interactive.html")
fig6.show()
print("Saved: chart6_interactive.html")


## 8. Summary

| Chart | Type | Key Message |
|-------|------|-------------|
| 1 | Line | Depression has risen globally since 1990 |
| 2 | Horizontal Bar | A few countries carry the heaviest burden |
| 3 | Heatmap | Burden is concentrated and growing |
| 4 | Multi-line | High-burden countries diverge over time |
| 5 | Slope Graph | Most countries worsened from 1990 to 2017 |
| 6 | Interactive Scatter | Depression & anxiety co-occur in high-burden countries |

**Files generated:** `chart1_global_trend.png`, `chart2_top_countries.png`, `chart3_heatmap.png`, `chart4_country_lines.png`, `chart5_slope_graph.png`, `chart6_interactive.html`
